# Worm locomotion analysis

Load Tierpsy speed CSVs, plot traces, and extract features with the
**centroid path-rate activity filter** (BL/s over a centered window on the
full blob track, mapped onto speed frames).

Inactive frames are excluded from active-filtered metrics.

**Reversals** use the sign-dwell filter: direction flips only after
`REVERSAL_DWELL_FRAMES` consecutive opposite-sign frames. Tracking gaps
reset direction state.

**Reorientation** (active reversals): pre vector is the most-extended
head→tail skeleton before the reversal. Post vector is the most-extended
skeleton within 1 s of the next forward-bout start that begins within 10 s
of reverse-bout end. Unscored if no such bout or no usable HT axis.
Counted when the angle exceeds `REORIENTATION_ANGLE_DEG` (default 90°).

Rates are also reported **per minute** (360 frames = 1 min at 6 fps).

Notebook plots use **interactive Plotly** widgets unless
`USE_INTERACTIVE_PLOTS = False` in the setup cell.

**Data:** read from the data root, by default the `Sample_data/` folder next to
`custom_python_scripts/` (one sample
video, `071026_HML1630_01`). Per-worm CSVs come from
`<data root>/Tierpsy_csvs/<strain>/`, the centroid track used for activity
from `<data root>/Videos/Results/<video>_featuresN.hdf5`, and per-video FPS
from `<data root>/video_fps.csv`. To use your own data, lay it out the same
way (see `Sample_data/README.md`) and set `DATA_ROOT` in the setup cell.
Strains with no CSV folder are skipped.

**Kernel:** `Python (worm-analysis)`

After changing filter knobs in the setup cell (or in `worm_analysis/metrics.py`),
**re-run the setup cell**, then recompute/plot.


In [ ]:
from pathlib import Path
import sys
import importlib

import matplotlib.pyplot as plt
import pandas as pd
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

# Reload so edits to filter constants in metrics.py take effect
import worm_analysis.metrics as metrics
import worm_analysis.plotting as plotting
import worm_analysis.io as io
importlib.reload(metrics)
importlib.reload(io)
import worm_analysis.video_fps as video_fps
importlib.reload(video_fps)
importlib.reload(plotting)
import worm_analysis.interactive as interactive
importlib.reload(interactive)

# --- Reversal sign-dwell filter ---
USE_REVERSAL_SUSTAIN_FILTER = True   # False → dwell=1 (immediate sign change)
REVERSAL_DWELL_FRAMES = 3           # consecutive opposite-sign frames to flip direction
metrics.USE_REVERSAL_SUSTAIN_FILTER = USE_REVERSAL_SUSTAIN_FILTER
metrics.REVERSAL_DWELL_FRAMES = REVERSAL_DWELL_FRAMES
metrics.REVERSAL_MIN_NEG_FRAMES = REVERSAL_DWELL_FRAMES  # back-compat alias
metrics.REVERSAL_MIN_POS_FRAMES = REVERSAL_DWELL_FRAMES  # back-compat alias



# --- Centroid BL/s activity knobs ---
CENTROID_ACTIVITY_THRESHOLD_BL_S = 0.1  # active if window path-rate > this
CENTROID_ACTIVITY_WINDOW_S = 15.0       # centered scanning window (seconds)
metrics.CENTROID_ACTIVITY_THRESHOLD_BL_S = CENTROID_ACTIVITY_THRESHOLD_BL_S
metrics.CENTROID_ACTIVITY_WINDOW_S = CENTROID_ACTIVITY_WINDOW_S

# --- Active-reversal reorientation ---
REORIENTATION_ANGLE_DEG = 60.0  # count when pre→post angle exceeds this
REORIENTATION_WINDOW_S = 3.0    # pre-reversal most-extended skeleton search (s)
REORIENTATION_FWD_SEARCH_S = 15.0  # next forward bout must start within this of reverse-bout end
REORIENTATION_FWD_BEST_S = 3.0     # most-extended HT within this of that forward start
metrics.REORIENTATION_ANGLE_DEG = REORIENTATION_ANGLE_DEG
metrics.REORIENTATION_WINDOW_S = REORIENTATION_WINDOW_S
metrics.REORIENTATION_FWD_SEARCH_S = REORIENTATION_FWD_SEARCH_S
metrics.REORIENTATION_FWD_BEST_S = REORIENTATION_FWD_BEST_S

# --- Continuous-minute extraction (worm-agnostic) ---
CONTINUOUS_SPEED_GAP_FRAMES = 36  # max gap (frames) still counted as continuous
metrics.CONTINUOUS_SPEED_GAP_FRAMES = CONTINUOUS_SPEED_GAP_FRAMES
USE_CONTINUOUS_MINUTE_BACKFILL = True  # overlapping end window when leftover ≥½ minute
metrics.USE_CONTINUOUS_MINUTE_BACKFILL = USE_CONTINUOUS_MINUTE_BACKFILL

from worm_analysis import (
    load_speed_csv,
    load_speed_directory,
    compute_metrics,
    compute_continuous_minute_metrics,
    apply_activity_filter,
    plot_worm_summary,
)
from worm_analysis.metrics import (
    summarize_worms,
    build_summary_tables,
    MIN_FRAMES,
    CONTINUOUS_SPEED_GAP_FRAMES,
    USE_CONTINUOUS_MINUTE_BACKFILL,
)
from worm_analysis.plotting import (
    plot_metrics_board,
)

# --- Interactive notebook plots (Plotly: zoom / pan / hover) ---
USE_INTERACTIVE_PLOTS = False
if USE_INTERACTIVE_PLOTS:
    plot_worm_summary = interactive.plot_worm_summary
    plot_metrics_board = interactive.plot_metrics_board


def show(fig):
    """Show Plotly widgets or matplotlib figures."""
    mod = getattr(type(fig), "__module__", "") or ""
    if "plotly" in mod:
        fig.show()
    else:
        plt.show()


# --- Data location ---
# Default: the Sample_data/ folder next to custom_python_scripts/. For your own data, lay it out like
# Sample_data/ (see Sample_data/README.md) and set DATA_ROOT to that folder.
DATA_ROOT = None  # e.g. "/path/to/my_data"
import worm_analysis.tierpsy_data as tierpsy_data
importlib.reload(tierpsy_data)
tierpsy_data.set_data_root(DATA_ROOT)

N2_CSV_DIR = tierpsy_data.strain_csv_dir("N2")
HML1641_CSV_DIR = tierpsy_data.strain_csv_dir("HML1641")
VM487_CSV_DIR = tierpsy_data.strain_csv_dir("VM487")
HML1630_CSV_DIR = tierpsy_data.strain_csv_dir("HML1630")
HML1660_CSV_DIR = tierpsy_data.strain_csv_dir("HML1660")
OUTPUT_DIR = ROOT / "output" / "notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_strain(csv_dir, dataset, pattern="*_fixed.csv"):
    """Load one strain's worms, or return {} (with a note) if it has no CSVs."""
    if not csv_dir.is_dir() or not any(csv_dir.glob(pattern)):
        print(f"{dataset}: no {pattern} files in {csv_dir} — skipped")
        return {}
    return load_speed_directory(
        csv_dir, pattern=pattern, fps=FPS, min_frames=MIN_FRAMES,
        fps_table=FPS_TABLE, dataset=dataset,
    )


# Per-video true FPS from video_fps.csv (run: python video_processing/build_video_fps_table.py)
from worm_analysis.video_fps import load_video_fps_table
FPS_TABLE_PATH = tierpsy_data.fps_table_path()
if FPS_TABLE_PATH.is_file():
    FPS_TABLE = load_video_fps_table(FPS_TABLE_PATH)
    print(f"Loaded FPS table: {FPS_TABLE_PATH} ({len(FPS_TABLE)} videos)")
else:
    FPS_TABLE = None
    print(f"WARNING: {FPS_TABLE_PATH} not found — using fallback FPS=6.0")
FPS = 6.0  # fallback only when a video is missing from the table

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
print("Data root:", tierpsy_data.data_root())
print("Plots:", "interactive Plotly (zoom/pan/hover)" if USE_INTERACTIVE_PLOTS else "static matplotlib")
print(
    f"Activity filter: CENTROID — path-rate > {metrics.CENTROID_ACTIVITY_THRESHOLD_BL_S:g} BL/s "
    f"over {metrics.CENTROID_ACTIVITY_WINDOW_S:g}s window (full blob track → speed frames)"
)
if metrics.USE_REVERSAL_SUSTAIN_FILTER:
    print(
        f"Reversal dwell: ON — flip after {metrics.REVERSAL_DWELL_FRAMES} "
        "opposite-sign frames (gaps reset direction state)"
    )
else:
    print("Reversal dwell: OFF — single opposite-sign frame flips direction")
print(
    f"Continuous minutes: gaps ≤{metrics.CONTINUOUS_SPEED_GAP_FRAMES} frames, "
    f"{metrics.FRAMES_PER_MINUTE} samples/min; "
    + ("backfill ON (≥½ window leftovers)" if metrics.USE_CONTINUOUS_MINUTE_BACKFILL else "backfill OFF")
    + " (recordings need a *_blob_perimeter.csv)"
)
print(
    f"Centroid BL/s knobs: threshold {metrics.CENTROID_ACTIVITY_THRESHOLD_BL_S:g} BL/s, "
    f"window {metrics.CENTROID_ACTIVITY_WINDOW_S:g}s "
    f"(centroid path-rate activity)"
)
print(
    f"Reorientation: angle > {metrics.REORIENTATION_ANGLE_DEG:g}°; "
    f"pre ±{metrics.REORIENTATION_WINDOW_S:g}s, "
    f"post HT within {metrics.REORIENTATION_FWD_BEST_S:g}s of next forward bout "
    f"(search {metrics.REORIENTATION_FWD_SEARCH_S:g}s after reverse end; unscored if none)"
)


In [ ]:
# N2
N2_DATA_DIR = N2_CSV_DIR
PATTERN = "*_fixed.csv"

N2_worms = load_strain(N2_DATA_DIR, "N2", pattern=PATTERN)
print(f"Loaded {len(N2_worms)} worms from {N2_DATA_DIR.name}:")
for wid, df in N2_worms.items():
    has_p = "blob_perimeter" in df.columns and df["blob_perimeter"].notna().any()
    fps_v = float(df["fps_true"].iloc[0]) if "fps_true" in df.columns else float("nan")
    fpm_v = int(df["frames_per_minute"].iloc[0]) if "frames_per_minute" in df.columns else 0
    print(f"  {wid:6s}  n={len(df):5d}  blob_perimeter={'yes' if has_p else 'NO'}  fps={fps_v:.3f}  fpm={fpm_v}")


In [ ]:
# VM487
VM487_DATA_DIR = VM487_CSV_DIR
PATTERN = "*_fixed.csv"

VM487_worms = load_strain(VM487_DATA_DIR, "VM487", pattern=PATTERN)
print(f"Loaded {len(VM487_worms)} worms from {VM487_DATA_DIR.name}:")
for wid, df in VM487_worms.items():
    has_p = "blob_perimeter" in df.columns and df["blob_perimeter"].notna().any()
    fps_v = float(df["fps_true"].iloc[0]) if "fps_true" in df.columns else float("nan")
    fpm_v = int(df["frames_per_minute"].iloc[0]) if "frames_per_minute" in df.columns else 0
    print(f"  {wid:6s}  n={len(df):5d}  blob_perimeter={'yes' if has_p else 'NO'}  fps={fps_v:.3f}  fpm={fpm_v}")


In [ ]:
# HML1641
HML1641_DATA_DIR = HML1641_CSV_DIR
PATTERN = "*_fixed.csv"

HML1641_worms = load_strain(HML1641_DATA_DIR, "HML1641", pattern=PATTERN)
print(f"Loaded {len(HML1641_worms)} worms from {HML1641_DATA_DIR.name}:")
for wid, df in HML1641_worms.items():
    has_p = "blob_perimeter" in df.columns and df["blob_perimeter"].notna().any()
    fps_v = float(df["fps_true"].iloc[0]) if "fps_true" in df.columns else float("nan")
    fpm_v = int(df["frames_per_minute"].iloc[0]) if "frames_per_minute" in df.columns else 0
    print(f"  {wid:6s}  n={len(df):5d}  blob_perimeter={'yes' if has_p else 'NO'}  fps={fps_v:.3f}  fpm={fpm_v}")


In [ ]:
# HML1630 
HML1630_DATA_DIR = HML1630_CSV_DIR
PATTERN = "*_fixed.csv"

HML1630_worms = load_strain(HML1630_DATA_DIR, "HML1630", pattern=PATTERN)
print(f"Loaded {len(HML1630_worms)} worms from {HML1630_DATA_DIR.name}:")
for wid, df in HML1630_worms.items():
    has_p = "blob_perimeter" in df.columns and df["blob_perimeter"].notna().any()
    fps_v = float(df["fps_true"].iloc[0]) if "fps_true" in df.columns else float("nan")
    fpm_v = int(df["frames_per_minute"].iloc[0]) if "frames_per_minute" in df.columns else 0
    print(f"  {wid:6s}  n={len(df):5d}  blob_perimeter={'yes' if has_p else 'NO'}  fps={fps_v:.3f}  fpm={fpm_v}")


In [ ]:
# HML1660

HML1660_DATA_DIR = HML1660_CSV_DIR
PATTERN = "*_fixed.csv"

HML1660_worms = load_strain(HML1660_DATA_DIR, "HML1660", pattern=PATTERN)
print(f"Loaded {len(HML1660_worms)} worms from {HML1660_DATA_DIR.name}:")
for wid, df in HML1660_worms.items():
    has_p = "blob_perimeter" in df.columns and df["blob_perimeter"].notna().any()
    fps_v = float(df["fps_true"].iloc[0]) if "fps_true" in df.columns else float("nan")
    fpm_v = int(df["frames_per_minute"].iloc[0]) if "frames_per_minute" in df.columns else 0
    print(f"  {wid:6s}  n={len(df):5d}  blob_perimeter={'yes' if has_p else 'NO'}  fps={fps_v:.3f}  fpm={fpm_v}")


## 2. Metrics for one worm (W4)


In [ ]:
# Must specify worms dataframe for dataset manually in line 3

WORM_ID = "W4"

df = HML1630_worms[WORM_ID]
result = compute_metrics(df)

fig = plot_metrics_board(result)
show(fig)






print(pd.Series(result.metrics.to_dict()).to_string())
print("\nPer-minute breakdown:")
display(result.per_minute)

active_frames = apply_activity_filter(result.frames)
print(f"Active frames: {len(active_frames)} / {len(result.frames)}")


## 2a. Speed traces — one worm per strain

Same speed panel as the top of `plot_metrics_board` in §2, stacked for one
example worm from each loaded strain. Legends are off. The x-axis is seconds
from t=0 to t=600 (full plot width). Set `sharey=False` for independent
y-axis scaling per strain. No activity background shading: inactive speed
is drawn in `INACTIVE_COLOR`, using the setup-cell activity filter
(centroid path-rate knobs in the setup cell). Yellow reversal markers and red/green
speed coloring use the same `compute_metrics` path as batch §3 (setup-cell
dwell and centroid activity). Reverse-state speed after a kept
reversal is muted red; Color flips at y=0.
Edit `FORWARD_COLOR`, `REVERSE_COLOR`, `INACTIVE_COLOR`, `MARKER_SHAPE`,
`MARKER_COLOR`, and `MARKER_SIZE` in the cell below, then re-run.
Re-running the cell writes EPS files (stack + one per strain) under
`output/notebook/speed_one_per_strain_eps_final_clean/`. Edit `EXAMPLE_WORM_IDS` to pin a worm
(default prefers `W4` when present).


In [ ]:
import importlib
import worm_analysis.explore as explore
import worm_analysis.plotting as plotting

importlib.reload(plotting)
importlib.reload(explore)
from worm_analysis.explore import plot_speed_one_worm_per_strain

# Optional: pin a worm id per strain (omit a strain to use the default picker).
EXAMPLE_WORM_IDS = {
    "N2": "W1",
    "VM487": "W3_3",
    "HML1641": "W4",
    "HML1630": "W5",
    "HML1660": "W9_2",
}

# --- style (edit, then re-run) ---
FORWARD_COLOR = "#000000"     # forward-bout speed
REVERSE_COLOR = "#B00000"     # reverse-bout speed
INACTIVE_COLOR = "#bfbfbf"    # inactive speed (setup-cell activity filter)
MARKER_SHAPE = "o"            # matplotlib marker: "o" circle, "|" tick, "^" triangle, "s" square, "D" diamond, "x"
MARKER_COLOR = "#ffdd33"
MARKER_SIZE = 24              # scatter size (points^2)


_candidates = {
    "N2": "N2_worms",
    "VM487": "VM487_worms",
    "HML1641": "HML1641_worms",
    "HML1630": "HML1630_worms",
    "HML1660": "HML1660_worms",
}
strain_worms = {}
for strain, var in _candidates.items():
    worms = globals().get(var)
    if isinstance(worms, dict) and worms:
        strain_worms[strain] = worms

EPS_DIR = OUTPUT_DIR / "speed_one_per_strain_eps_final_clean"

fig, example_worms = plot_speed_one_worm_per_strain(
    strain_worms,
    worm_ids=EXAMPLE_WORM_IDS,
    show_legend=False,
    sharex=True,
    sharey=False,
    eps_dir=EPS_DIR,
    line_color=FORWARD_COLOR,
    reverse_color=REVERSE_COLOR,
    inactive_color=INACTIVE_COLOR,
    reversal_marker=MARKER_SHAPE,
    reversal_color=MARKER_COLOR,
    reversal_marker_size=MARKER_SIZE,
)
display(example_worms)
print(f"EPS exports → {EPS_DIR}")
show(fig)


## 3. Batch metrics


In [ ]:
datasets = ['N2_worms', 'VM487_worms', 'HML1641_worms', 'HML1630_worms', 'HML1660_worms']
datasets = [name for name in datasets if globals().get(name)]  # skip strains with no data
print("strains with data:", datasets)
dataset_tables = dict()
for i in datasets:
    x = eval(i)
    results = {}
    n_total = len(x)
    for k, (wid, wdf) in enumerate(x.items(), start=1):
        print(f"{i}: worm {k}/{n_total} {wid} …", flush=True)
        results[wid] = compute_metrics(wdf)
        print(f"{i}: worm {k}/{n_total} {wid} done  pct_active={results[wid].metrics.pct_active}", flush=True)

    tables = build_summary_tables(list(results.values()))
    dataset_tables[f'{i}_general'] = tables["general"]
    dataset_tables[f'{i}_active'] = tables["active"]
    dataset_tables[f'{i}_general_per_duration'] = tables["general_per_duration"]
    dataset_tables[f'{i}_active_per_active_duration'] = tables["active_per_active_duration"]
    dataset_tables[f'{i}_active_per_duration'] = tables["active_per_duration"]
    print(f"{i}: finished {len(results)} worms", flush=True)


# Worm-agnostic continuous minutes (eligible fixed CSVs with full sibling set)
for i in datasets:
    x = eval(i)
    print(f"{i}: continuous minutes …", flush=True)
    _results, minute_tables = compute_continuous_minute_metrics(x)
    dataset_tables[f'{i}_continuous_minutes_meta'] = minute_tables['minute_meta']
    dataset_tables[f'{i}_continuous_minutes_general'] = minute_tables['general']
    dataset_tables[f'{i}_continuous_minutes_active'] = minute_tables['active']
    dataset_tables[f'{i}_continuous_minutes_general_per_duration'] = minute_tables['general_per_duration']
    dataset_tables[f'{i}_continuous_minutes_active_per_duration'] = minute_tables['active_per_duration']
    dataset_tables[f'{i}_continuous_minutes_active_per_active_duration'] = minute_tables['active_per_active_duration']
    print(f"{i}: {len(minute_tables['minute_meta'])} continuous minutes", flush=True)


In [ ]:
# Build worm-level and continuous-minute details tables (join on id).
# Specs live in worm_analysis/explore.py — edit WORM_DETAILS_SPEC / MINUTE_DETAILS_SPEC there
# or pass a custom spec to build_strain_details / build_minute_details.
import importlib
import worm_analysis.explore as explore
importlib.reload(explore)
from worm_analysis.explore import (
    build_all_details,
    build_strain_details,
    build_minute_details,
    WORM_DETAILS_SPEC,
    MINUTE_DETAILS_SPEC,
)

build_all_details(
    dataset_tables,
    datasets,
    include_worms=True,
    include_minutes=True,
)


In [ ]:
display(dataset_tables[f'{datasets[0]}_details'].describe())

## 4. Compile one metric across strains

Pick a table type (`general`, `active`, `general_per_duration`, `active_per_duration`, or `active_per_active_duration`) and a column name. Values for each strain are written as separate columns in one CSV (unequal worm counts are padded with blank cells).

In [ ]:
def compile_metric_across_strains(
    dataset_tables,
    *,
    table="details",
    column="active_reversals_per_active_min",
    datasets=None,
    include_worm_ids=False,
    output_path=None,
):
    """
    Build a CSV with one column per strain for a single metric.

    Parameters
    ----------
    dataset_tables : dict
        Keys like 'N2_worms_general', 'VM487_worms_active', ...
    table : str
        Table suffix: general | active | general_per_duration |
        active_per_duration | active_per_active_duration
    column : str
        Metric column to pull (e.g. 'reversals', 'active_reversals_per_min')
    datasets : list[str] or None
        Dataset keys without the table suffix. Defaults to the notebook
        ``datasets`` list (N2_worms, VM487_worms, ...).
    include_worm_ids : bool
        If True, also write a ``<strain>_worm_id`` column beside each metric.
    output_path : path-like or None
        Where to save. Default: OUTPUT_DIR / ``{column}_by_strain_{table}.csv``
    """
    if datasets is None:
        datasets = globals().get(
            "datasets",
            ["N2_worms", "VM487_worms", "HML1641_worms", "HML1630_worms", "HML1660_worms"],
        )

    series = {}
    missing = []
    for ds in datasets:
        key = f"{ds}_{table}"
        if key not in dataset_tables:
            missing.append(key)
            continue
        df = dataset_tables[key]
        if column not in df.columns:
            raise KeyError(
                f"Column {column!r} not in {key}. "
                f"Available: {list(df.columns)}"
            )
        strain = ds.removesuffix("_worms")
        metric = df[column].reset_index(drop=True)
        if include_worm_ids and "worm_id" in df.columns:
            series[f"{strain}_worm_id"] = df["worm_id"].reset_index(drop=True)
        series[strain] = metric

    if missing:
        raise KeyError(
            "Missing tables in dataset_tables: "
            + ", ".join(missing)
            + f". Have: {sorted(dataset_tables)}"
        )
    if not series:
        raise ValueError("No data compiled — check datasets / table name.")

    out = pd.DataFrame(series)
    if output_path is None:
        output_path = OUTPUT_DIR / f"{column}_by_strain_{table}.csv"
    else:
        output_path = Path(output_path)
    out.to_csv(output_path, index=False)
    print(f"Wrote {output_path}  ({out.shape[0]} rows × {out.shape[1]} cols)")
    return out


# --- edit these ---
TABLE = "details"          # e.g. general, active, active_per_duration
COLUMN = "avg_active_rev_bout_s"       # e.g. reversals, active_reversals, pct_pos

compiled = compile_metric_across_strains(
    dataset_tables,
    table=TABLE,
    column=COLUMN,
    datasets=datasets,
    include_worm_ids=False,
)
display(compiled)

## 4b. Across-strain bar plots (worm or minute)

Compare one metric across strains as mean ± SEM bars (with points).

1. Run batch metrics (§3) and `build_all_details` first.
2. Set `UNIT` to `"worm"` or `"minute"`, and `COLUMN` to any numeric summary column.
3. Set `SHOW_STATS = True` to run pairwise tests and draw significance brackets
   (`*` p<0.05, `**` <0.01, `***` <0.001, `****` <0.0001).
4. Choose `STATS_TEST`:
   - `"mannwhitney"` / `"nonparametric"` — Mann–Whitney U
   - `"welch"` / `"parametric"` — Welch t-test (unequal variance)
   - `"student"` — Student t-test (equal variance)
   - `"brunner_munzel"` — Brunner–Munzel
   - `"ks"` — two-sample Kolmogorov–Smirnov
5. Optional `P_ADJUST = "holm"` / `"bonferroni"` / `"benjamini_hochberg"` (`"bh"` / `"fdr"`) / `"none"`.


In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
display(dataset_tables[f'{datasets[0]}_active'].describe())

In [ ]:
# Across-strain comparison — edit UNIT / COLUMN / SHOW_STATS, then run.
# Requires §3 batch metrics. Prefer running build_all_details first,
# but raw summary tables work too via TABLE=.

import importlib
import worm_analysis.explore as explore
importlib.reload(explore)
from worm_analysis.explore import (
    collect_metric_across_strains,
    plot_metric_across_strains,
    plot_metrics_across_strains,
    WORM_DETAILS_SPEC
)

# --- edit these ---
UNIT = "worm"  # "worm" | "minute"
COLUMN = "avg_active_rev_bout_s"
TABLE = "details"  # fallback if details lack COLUMN
KIND = "bar"   # "bar" | "box" | "violin"
ERROR = "sem"  # bar only: "sem" | "sd" | "ci95"

SHOW_STATS = True
# "mannwhitney" | "nonparametric" | "welch" | "parametric" | "student" | "brunner_munzel" | "ks"
STATS_TEST = "welch"
P_ADJUST = "bh"          # "none" | "holm" | "bonferroni" | "benjamini_hochberg" ("bh" / "fdr")
ANNOTATE_NS = False        # True → also draw "ns" brackets

res = plot_metric_across_strains(
    dataset_tables,
    COLUMN,
    unit=UNIT,
    table=TABLE,
    datasets=datasets,
    kind=KIND,
    error=ERROR,
    stats=SHOW_STATS,
    stats_test=STATS_TEST,
    p_adjust=P_ADJUST,
    annotate_ns=ANNOTATE_NS,
)
plt.show()
display(res["summary"])
if SHOW_STATS and res.get("stats") is not None and len(res["stats"]):
    cols = ["test", "level_a", "level_b", "n_a", "n_b", "statistic", "p", "p_adj", "stars"]
    display(res["stats"][[c for c in cols if c in res["stats"].columns]])

skip = {"worm_id", "recording_date", "video_number", "minute_id", "source_worm_id"}
details_columns = [c for c, _ in WORM_DETAILS_SPEC if c not in skip]
data_for_paper = ['max_active_reversals_per_minute', 
     'active_reversals_per_active_min', 'avg_active_fwd_bout_s', 'active_reversals_reorient', 'active_reversals_reorient_per_active_min', 'avg_active_rev_bout_s']
# Optional: several metrics in one row
panel = plot_metrics_across_strains(
    dataset_tables,
    data_for_paper,
    unit=UNIT,
    table=TABLE,
    datasets=datasets,
    kind=KIND,
    stats=SHOW_STATS,
    stats_test=STATS_TEST,
    p_adjust=P_ADJUST,
)
# plt.tight_layout()
plt.show()